# Morocco Dataset Notebook

This notebook downloads the Morocco crop dataset files, creates one raw merged table, prepares the selected crop-plus-background Sentinel-2 time-series dataset, and builds one YieldSAT-aligned `features.parquet` file for merged crop-classification experiments. The enrichment reuses the existing Morocco band/NDVI CSVs, adds parcel geometry metadata, and fetches lightweight weather features by survey zone; expensive SCL/cloud-quality extraction is intentionally left out of the main run.


## Dependencies

This cell installs the small packages needed to read shapefile attributes and geometry, compute parcel area/centroids, save Parquet outputs, and call the weather API.


In [1]:
%pip install -q dbfread pyshp shapely pyproj requests pyarrow



## Download The Raw Figshare Files

This cell downloads only the two small dataset files needed for this workflow: the shapefile labels and the Sentinel-2/NDVI parcel time series. The large photo archive is intentionally skipped.

In [2]:
import requests
from pathlib import Path

ARTICLE_ID = 28486022
FILE_IDS = {
    56267117,  # Shapefile_Crop and irrigation dataset.rar
    56267120,  # Sentinel2 bands and Ndvi for surveyed parcels.rar
}

DATA_DIR = Path("data/morocco_crop_irrigation")
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/files").json()

for file_info in files:
    if file_info["id"] not in FILE_IDS:
        continue

    out_path = DATA_DIR / file_info["name"]
    print(f"Downloading: {file_info['name']}")

    with requests.get(file_info["download_url"], stream=True) as response:
        response.raise_for_status()
        with out_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024 * 8):
                if chunk:
                    output.write(chunk)

    print(f"Saved: {out_path}")


Downloading: Sentinel2 bands and Ndvi for surveyed parcels.rar
Saved: data/morocco_crop_irrigation/Sentinel2 bands and Ndvi for surveyed parcels.rar
Downloading: Shapefile_Crop and irrigation dataset.rar
Saved: data/morocco_crop_irrigation/Shapefile_Crop and irrigation dataset.rar


## Extract The Raw Files

This cell extracts the downloaded raw archives into the same working folder. It is intentionally written as simple shell commands for a Colab-style run.

In [3]:
# !apt-get update -qq
# !apt-get install -y unrar

In [4]:
!unrar x -idq "data/morocco_crop_irrigation/Shapefile_Crop and irrigation dataset.rar" "data/morocco_crop_irrigation/"

!unrar x -idq "data/morocco_crop_irrigation/Sentinel2 bands and Ndvi for surveyed parcels.rar" "data/morocco_crop_irrigation/"

## Load The Raw Dataset Into Memory

This cell reads the raw shapefile attributes and all raw Sentinel-2 parcel CSV files, then joins them by `parcel_id` in memory. It does not clean, filter, or save anything yet.


In [5]:
from pathlib import Path

import pandas as pd
from dbfread import DBF

DATA_DIR = Path("data/morocco_crop_irrigation")
SHAPE_DIR = DATA_DIR / "Shapefile_Crop and irrigation dataset"
S2_DIR = DATA_DIR / "Sentinel2 bands for all surveyed parcels"

In [6]:


raw_attributes = pd.DataFrame(
    iter(DBF(str(SHAPE_DIR / "lulc_fielddata.dbf"), encoding="utf-8"))
)

frames = []
for csv_path in sorted(S2_DIR.rglob("*_S2_timeseries.csv")):
    if ".ipynb_checkpoints" in csv_path.parts:
        continue

    parcel_id = csv_path.name.removesuffix("_S2_timeseries.csv")
    part = pd.read_csv(csv_path)
    part.insert(0, "parcel_id", parcel_id)
    part.insert(1, "raw_folder", csv_path.parent.name)
    part.insert(2, "raw_file", str(csv_path.relative_to(S2_DIR)))
    frames.append(part)

raw_sentinel2 = pd.concat(frames, ignore_index=True)
raw_merged = raw_sentinel2.merge(
    raw_attributes,
    on="parcel_id",
    how="left",
    suffixes=("_s2", "_label"),
)

print("Raw shapefile attributes:", raw_attributes.shape)
print("Raw Sentinel-2 time series:", raw_sentinel2.shape)
print("Raw merged table:", raw_merged.shape)


Raw shapefile attributes: (10383, 13)
Raw Sentinel-2 time series: (518907, 17)
Raw merged table: (518907, 29)


In [7]:
raw_checkpoint_path = DATA_DIR / "raw_merged.parquet"
raw_merged.to_parquet(raw_checkpoint_path, index=False)
raw_merged = pd.read_parquet(raw_checkpoint_path)

print("Saved and reloaded raw checkpoint:", raw_checkpoint_path)
print("Raw merged shape:", raw_merged.shape)


Saved and reloaded raw checkpoint: data/morocco_crop_irrigation/raw_merged.parquet
Raw merged shape: (518907, 29)


## Explore The Raw Merged Table

From this point onward, the notebook works only with `raw_merged`. This keeps the workflow simple: the raw shapefile attributes and Sentinel-2 time series were already merged and saved as one file.

In [8]:
raw_parcels = raw_merged.drop_duplicates("parcel_id").copy()

print("Raw merged shape:", raw_merged.shape)
print("Unique parcels:", raw_merged["parcel_id"].nunique())
print("Columns:")
print(raw_merged.columns.tolist())

print("\nRaw parcel categories:")
print(raw_parcels["crop"].value_counts(dropna=False))

print("\nTop seasonal crop labels:")
print(raw_parcels["season"].value_counts(dropna=False).head(25))

print("\nIrrigation values:")
print(raw_parcels["irrigation"].value_counts(dropna=False))

print("\nSurvey zones:")
print(raw_parcels["zone"].value_counts(dropna=False))

Raw merged shape: (518907, 29)
Unique parcels: 9993
Columns:
['parcel_id', 'raw_folder', 'raw_file', 'B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'NDVI', 'date_s2', 'date_label', 'crop', 'season', 'tree', 'irrigation', 'btw_lines', 'btw_trees', 'height', 'diameter', 'cp_height', 'zone', 'photo']

Raw parcel categories:
crop
Seasonal Crop       6890
Trees               1425
Bare soil            942
Trees + Seasonal     374
Weed                 346
Water                 16
Name: count, dtype: int64

Top seasonal crop labels:
season
Wheat          2768
               2716
Corn           1268
Alfalfa         912
Potatoes        441
Beets           425
Peas            252
Peanut          180
Onions          176
Tomatoes        128
Sugarcane        94
Pumpkins         92
Fava Beans       54
Watermelon       51
Melons           51
Pepper           47
Eggplant         43
Cauliflower      38
Cabbage          33
Parsley          32
Carrots          26
Zucchini      

## Keep Reviewed Crops And Background Parcels

For this cleaned dataset, we keep the three main target crops (`Wheat`, `Corn`, `Alfalfa`), ten reviewed seasonal crops, and the three background classes (`Bare soil`, `Water`, `Weed`). The ten reviewed crops are kept because they have enough samples for this version (`count >= 25`) and are agriculturally close to one of the three main target crops, so they are useful candidates for a later `Other` group instead of being discarded immediately.

The reviewed crop relationships are:

| Future group direction | Reviewed crops kept for later |
|---|---|
| Close to `Wheat` | `Beets`, `Potatoes`, `Peas`, `Onions`, `Fava Beans` |
| Close to `Corn` | `Peanut`, `Tomatoes`, `Sugarcane`, `Rice` |
| Close to `Alfalfa` | `Peas`, `Peanut`, `Fava Beans`, `Parsley` |

Some crops can support more than one future group, such as `Peas`, `Peanut`, and `Fava Beans`. We do not group them here. They stay as original labels so a later modeling notebook can decide the final `Other` strategy.

Tree-only and mixed tree-seasonal parcels are removed because their satellite signal is not a clean seasonal field-crop signal.


In [9]:
main_crops = ["Wheat", "Corn", "Alfalfa"]

other_candidate_crops = [
    "Beets",
    "Potatoes",
    "Peas",
    "Peanut",
    "Onions",
    "Tomatoes",
    "Sugarcane",
    "Fava Beans",
    "Parsley",
    "Rice",
]

background_classes = ["Bare soil", "Water", "Weed"]
kept_seasonal_crops = main_crops + other_candidate_crops

crop_similarity_groups = {
    "Wheat": ["Barley", "Fava Beans", "Peas", "Green Peas", "Beets", "Onions", "Potatoes"],
    "Corn": ["Sorghum", "Sugarcane", "Rice", "Sunflower", "Peanut", "Beans", "Tomatoes"],
    "Alfalfa": ["Fava Beans", "Peas", "Green Peas", "Beans", "Peanut", "Mint", "Parsley"],
}

selected_merged = raw_merged[
    (
        (raw_merged["crop"] == "Seasonal Crop")
        & (raw_merged["season"].isin(kept_seasonal_crops))
    )
    | (raw_merged["crop"].isin(background_classes))
].copy()

selected_merged["label"] = selected_merged.apply(
    lambda row: row["season"] if row["crop"] == "Seasonal Crop" else row["crop"],
    axis=1,
)

selected_merged = selected_merged.rename(
    columns={
        "date_s2": "satellite_date",
        "date_label": "survey_date",
        "zone": "survey_zone",
        "NDVI": "ndvi",
    }
)

selected_merged["irrigation_type"] = selected_merged["irrigation"].astype(str).str.strip()
selected_merged["irrigation_type"] = selected_merged["irrigation_type"].replace(
    {
        "rainfed": "Rainfed",
        "Flood IrrigationFlood Irrigation": "Flood Irrigation",
        "": None,
        "nan": None,
        "None": None,
    }
)

selected_parcel_labels = selected_merged[["parcel_id", "label"]].drop_duplicates()

print("Main crops:", main_crops)
print("Future Other candidate crops:", other_candidate_crops)
print("Background classes:", background_classes)
print("Selected crop/background time-series shape:", selected_merged.shape)
print("Selected parcels:", selected_merged["parcel_id"].nunique())
print("\nSelected label counts:")
print(selected_parcel_labels["label"].value_counts(dropna=False))


Main crops: ['Wheat', 'Corn', 'Alfalfa']
Future Other candidate crops: ['Beets', 'Potatoes', 'Peas', 'Peanut', 'Onions', 'Tomatoes', 'Sugarcane', 'Fava Beans', 'Parsley', 'Rice']
Background classes: ['Bare soil', 'Water', 'Weed']
Selected crop/background time-series shape: (390766, 31)
Selected parcels: 7734

Selected label counts:
label
Wheat         2614
Corn          1252
Bare soil      942
Alfalfa        857
Beets          424
Potatoes       416
Weed           346
Peas           229
Peanut         167
Onions         149
Tomatoes       128
Sugarcane       94
Fava Beans      44
Parsley         31
Rice            25
Water           16
Name: count, dtype: int64


## Compact Dataset Checks

These checks confirm the cleaned crop/background dataset shape, label distribution, irrigation distribution, zone distribution, and Sentinel-2 coverage. The dataset is still time-series data, so one parcel appears on multiple satellite observation dates.


In [10]:
selected_parcels = selected_merged.drop_duplicates("parcel_id")

ts_coverage = (
    selected_merged
    .assign(satellite_date=pd.to_datetime(selected_merged["satellite_date"], errors="coerce"))
    .groupby("parcel_id")
    .agg(
        observation_count=("satellite_date", "count"),
        first_observation=("satellite_date", "min"),
        last_observation=("satellite_date", "max"),
    )
)
ts_coverage["time_span_days"] = (
    ts_coverage["last_observation"] - ts_coverage["first_observation"]
).dt.days

print("Label counts:")
display(selected_parcels["label"].value_counts(dropna=False).to_frame("parcel_count"))

print("Irrigation counts:")
display(selected_parcels["irrigation_type"].value_counts(dropna=False).to_frame("parcel_count"))

print("Zone counts:")
display(selected_parcels["survey_zone"].value_counts(dropna=False).to_frame("parcel_count"))

print("Sentinel-2 coverage summary:")
display(ts_coverage[["observation_count", "time_span_days"]].describe().loc[["min", "25%", "50%", "75%", "max"]])


Label counts:


,parcel_count
label,
Wheat,2614
Corn,1252
Bare soil,942
Alfalfa,857
Beets,424
Potatoes,416
Weed,346
Peas,229
Peanut,167


Irrigation counts:


,parcel_count
irrigation_type,
Flood Irrigation,2461
Drip Irrigation,1391
None,1274
Sprinkler Irrigation,1215
Rainfed,1107
Basin Irrigation,254
Pivot Irrigation,32


Zone counts:


,parcel_count
survey_zone,
Doukkala,3948
Gharb,1292
Tadla,1161
el haouz,918
Statte,248
Souss,167


Sentinel-2 coverage summary:


,observation_count,time_span_days
min,18.0,305.0
25%,33.0,330.0
50%,35.0,353.0
75%,65.0,358.0
max,301.0,360.0


## Coverage And Cleaning Conclusion

The cleaned output keeps reviewed seasonal crop labels plus `Bare soil`, `Water`, and `Weed` as background classes. It includes the three main crops (`Wheat`, `Corn`, `Alfalfa`) and ten reviewed crops that can later be considered for an `Other` class.

The Sentinel-2 coverage is broad across 2024, but the number of observations per parcel is uneven, so later modeling should use aggregation, time-window features, interpolation, or a sequence model that can handle variable-length inputs.

The output does not keep `source_folder`, `source_file`, raw irrigation text, photo paths, tree labels, mixed tree-seasonal labels, tree-only labels, unreviewed rare crops, or tree-structure measurement columns.


## Save One Selected Crop/Background Time-Series File

This checkpoint saves one easy-load Parquet file for the next notebook. The file is still a time-series table: one row per parcel per Sentinel-2 observation date.


In [11]:
final_columns = [
    "parcel_id",
    "satellite_date",
    "B1",
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B9",
    "B11",
    "B12",
    "ndvi",
    "survey_date",
    "label",
    "irrigation_type",
    "survey_zone",
]

final_dataset = selected_merged[final_columns].copy()
output_path = DATA_DIR / "morocco_selected_crop_background_timeseries.parquet"
final_dataset.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Final selected crop/background time-series shape:", final_dataset.shape)
print("Final parcels:", final_dataset["parcel_id"].nunique())
print("Final columns:")
print(final_dataset.columns.tolist())


Saved: data/morocco_crop_irrigation/morocco_selected_crop_background_timeseries.parquet
Final selected crop/background time-series shape: (390766, 19)
Final parcels: 7734
Final columns:
['parcel_id', 'satellite_date', 'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'ndvi', 'survey_date', 'label', 'irrigation_type', 'survey_zone']


## Build Parcel Geometry Metadata

The Figshare shapefile includes real WGS84 parcel polygons. This section extracts one geometry row per parcel with WKT, centroid, bounding box, and approximate area so later enrichment can use weather centroids, geometry diagnostics, or optional external quality checks without duplicating geometry on every time-series row.


In [12]:
import math
import numpy as np
import shapefile
from pyproj import Transformer
from shapely.geometry import MultiPolygon, Polygon
from shapely.ops import transform as shapely_transform

SHAPE_PATH = SHAPE_DIR / "lulc_fielddata.shp"


def shape_to_geometry(shape):
    if not shape.points:
        return None

    breaks = list(shape.parts) + [len(shape.points)]
    polygons = []
    for start, end in zip(breaks[:-1], breaks[1:]):
        ring = shape.points[start:end]
        if len(ring) < 4:
            continue
        polygon = Polygon(ring)
        if not polygon.is_empty and polygon.area > 0:
            polygons.append(polygon)

    if not polygons:
        return None
    if len(polygons) == 1:
        return polygons[0]
    return MultiPolygon(polygons)


def estimate_utm_epsg(lon, lat):
    zone = int((lon + 180) / 6) + 1
    return 32600 + zone if lat >= 0 else 32700 + zone


def projected_area_ha(geometry):
    if geometry is None or geometry.is_empty:
        return np.nan
    centroid = geometry.centroid
    epsg = estimate_utm_epsg(centroid.x, centroid.y)
    transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    projected = shapely_transform(transformer.transform, geometry)
    return float(projected.area / 10000.0)


reader = shapefile.Reader(str(SHAPE_PATH), encoding="utf-8")
shape_fields = [field[0] for field in reader.fields[1:]]
geometry_rows = []

for record, shape in zip(reader.records(), reader.shapes()):
    attrs = dict(zip(shape_fields, record))
    geometry = shape_to_geometry(shape)
    centroid = geometry.centroid if geometry is not None else None
    bbox = shape.bbox if shape.points else [np.nan, np.nan, np.nan, np.nan]

    geometry_rows.append(
        {
            "parcel_id": attrs.get("parcel_id"),
            "geometry_wkt": geometry.wkt if geometry is not None else None,
            "centroid_lon": float(centroid.x) if centroid is not None else np.nan,
            "centroid_lat": float(centroid.y) if centroid is not None else np.nan,
            "bbox_min_lon": float(bbox[0]),
            "bbox_min_lat": float(bbox[1]),
            "bbox_max_lon": float(bbox[2]),
            "bbox_max_lat": float(bbox[3]),
            "area_ha": projected_area_ha(geometry),
        }
    )

parcel_geometry = pd.DataFrame(geometry_rows)

print("Geometry rows:", parcel_geometry.shape)
print("Missing geometries:", parcel_geometry["geometry_wkt"].isna().sum())
print("CRS: EPSG:4326 / WGS84")
display(parcel_geometry[["area_ha", "centroid_lon", "centroid_lat"]].describe())


Geometry rows: (10383, 9)
Missing geometries: 1
CRS: EPSG:4326 / WGS84


,area_ha,centroid_lon,centroid_lat
count,10382.000000,10382.000000,10382.000000
mean,1.749719,-7.720669,32.748391
std,4.018546,1.041711,1.051325
min,0.025664,-9.465830,30.251787
25%,0.468382,-8.588002,32.267576
50%,0.843236,-8.158098,32.586720
75%,1.709004,-6.630699,33.164031
max,206.797069,-5.647853,35.172282


## Build Selected Parcel Metadata

This section creates a field-level Morocco table aligned with the YieldSAT feature-store idea. It keeps the original label, adds the five-class model label, and keeps geometry-derived fields for diagnostics and enrichment.

In [13]:
def make_five_class_label(label):
    if label in main_crops:
        return label
    if label in background_classes:
        return "Background"
    return "Other crop"


selected_parcel_metadata = (
    final_dataset[["parcel_id", "label", "survey_date", "irrigation_type", "survey_zone"]]
    .drop_duplicates("parcel_id")
    .merge(parcel_geometry, on="parcel_id", how="left")
)
selected_parcel_metadata["model_label"] = selected_parcel_metadata["label"].map(make_five_class_label)
selected_parcel_metadata["country"] = "Morocco"
selected_parcel_metadata["year"] = 2024
selected_parcel_metadata["dataset_source"] = "morocco_crop_irrigation"
selected_parcel_metadata["field_id"] = selected_parcel_metadata["parcel_id"].astype(str)

print("Selected fields:", selected_parcel_metadata.shape)
print("Missing selected geometries:", selected_parcel_metadata["geometry_wkt"].isna().sum())
print("Five-class label counts:")
display(selected_parcel_metadata["model_label"].value_counts().to_frame("parcel_count"))
print("Area summary in hectares:")
display(selected_parcel_metadata["area_ha"].describe().to_frame("area_ha"))


Selected fields: (7734, 18)
Missing selected geometries: 0
Five-class label counts:


,parcel_count
model_label,
Wheat,2614
Other crop,1707
Background,1304
Corn,1252
Alfalfa,857


Area summary in hectares:


,area_ha
count,7734.000000
mean,1.400335
std,2.447707
min,0.025664
25%,0.442146
50%,0.760499
75%,1.432555
max,57.692464


## Create YieldSAT-Aligned Observation Features

This section converts the Morocco time-series rows into YieldSAT-style observation features. It renames bands to `B01`/`B02` style, normalizes reflectance scale, computes vegetation indices available from the existing bands, and uses `s2_*_mean` names so the final merged notebook can aggregate both datasets consistently.

In [14]:
BAND_RENAME = {
    "B1": "B01",
    "B2": "B02",
    "B3": "B03",
    "B4": "B04",
    "B5": "B05",
    "B6": "B06",
    "B7": "B07",
    "B8": "B08",
    "B8A": "B8A",
    "B9": "B09",
    "B11": "B11",
    "B12": "B12",
}
S2_BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
VEGETATION_INDEX_NAMES = ["NDVI", "SAVI", "EVI", "GNDVI", "NDWI_GREEN", "NDRE", "NDMI", "MSI", "NBR", "BSI", "CAI_B01_B08"]


def safe_divide_series(numerator, denominator, min_abs=1e-10):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    output = numerator / denominator.where(denominator.abs() > min_abs)
    return output.replace([np.inf, -np.inf], np.nan)


def normalize_reflectance_series(values):
    values = pd.to_numeric(values, errors="coerce")
    high_value = values.dropna().quantile(0.99) if values.notna().any() else np.nan
    if pd.notna(high_value) and high_value > 2.0:
        values = values / 10000.0
    return values.where(values > 0)


aligned_source = final_dataset.copy()
aligned_source["date"] = pd.to_datetime(aligned_source["satellite_date"], errors="coerce")
aligned_source["doy"] = aligned_source["date"].dt.dayofyear

for source_col, target_col in BAND_RENAME.items():
    aligned_source[target_col] = normalize_reflectance_series(aligned_source[source_col])

b01 = aligned_source["B01"]
b02 = aligned_source["B02"]
b03 = aligned_source["B03"]
b04 = aligned_source["B04"]
b05 = aligned_source["B05"]
b08 = aligned_source["B08"]
b11 = aligned_source["B11"]
b12 = aligned_source["B12"]

computed_ndvi = safe_divide_series(b08 - b04, b08 + b04)
source_ndvi = pd.to_numeric(aligned_source["ndvi"], errors="coerce")
aligned_source["NDVI"] = source_ndvi.where(source_ndvi.between(-1.5, 1.5), computed_ndvi).fillna(computed_ndvi)
aligned_source["SAVI"] = safe_divide_series(1.5 * (b08 - b04), b08 + b04 + 0.5)
aligned_source["EVI"] = safe_divide_series(2.5 * (b08 - b04), b08 + 6.0 * b04 - 7.5 * b02 + 1.0)
aligned_source["GNDVI"] = safe_divide_series(b08 - b03, b08 + b03)
aligned_source["NDWI_GREEN"] = safe_divide_series(b03 - b08, b03 + b08)
aligned_source["NDRE"] = safe_divide_series(b08 - b05, b08 + b05)
aligned_source["NDMI"] = safe_divide_series(b08 - b11, b08 + b11)
aligned_source["MSI"] = safe_divide_series(b11, b08)
aligned_source["NBR"] = safe_divide_series(b08 - b12, b08 + b12)
aligned_source["BSI"] = safe_divide_series((b11 + b04) - (b08 + b02), (b11 + b04) + (b08 + b02))
aligned_source["CAI_B01_B08"] = safe_divide_series(b08 - b01, b08 + b01)

aligned_observations = aligned_source[["parcel_id", "date", "doy", "survey_zone"]].copy()
aligned_observations["field_id"] = aligned_observations["parcel_id"].astype(str)
aligned_observations["date"] = aligned_observations["date"].dt.date.astype(str)
aligned_observations["dataset_source"] = "morocco_crop_irrigation"

for band in S2_BANDS:
    aligned_observations[f"s2_{band}_mean"] = aligned_source[band]
for index_name in VEGETATION_INDEX_NAMES:
    aligned_observations[f"s2_{index_name}_mean"] = aligned_source[index_name]

ordered_columns = ["field_id", "parcel_id", "date", "doy", "dataset_source", "survey_zone"]
ordered_columns += [f"s2_{band}_mean" for band in S2_BANDS]
ordered_columns += [f"s2_{index_name}_mean" for index_name in VEGETATION_INDEX_NAMES]
aligned_observations = aligned_observations[ordered_columns].replace([np.inf, -np.inf], np.nan)

print("Aligned observation rows:", aligned_observations.shape)
print("Aligned observation feature columns:", len(ordered_columns) - 6)
print("Reflectance range check:")
display(aligned_observations[[f"s2_{band}_mean" for band in S2_BANDS]].describe().loc[["min", "50%", "max"]])
print("Vegetation index range check:")
display(aligned_observations[[f"s2_{name}_mean" for name in VEGETATION_INDEX_NAMES]].describe().loc[["min", "50%", "max"]])

Aligned observation rows: (390766, 29)
Aligned observation feature columns: 23
Reflectance range check:


,s2_B01_mean,s2_B02_mean,s2_B03_mean,s2_B04_mean,s2_B05_mean,s2_B06_mean,s2_B07_mean,s2_B08_mean,s2_B8A_mean,s2_B09_mean,s2_B11_mean,s2_B12_mean
min,1.557754e-07,7.690839e-07,0.000001,0.000129,0.004443,0.011786,0.014532,0.013692,0.015858,0.000013,0.019293,0.011872
50%,5.416707e-02,6.300468e-02,0.092175,0.123046,0.155759,0.232533,0.256418,0.267718,0.279087,0.283658,0.309411,0.257229
max,1.540288e+00,1.417600e+00,1.217447,1.150015,1.221457,1.139136,1.105278,1.189903,1.087519,1.713827,0.987389,0.851088


Vegetation index range check:


,s2_NDVI_mean,s2_SAVI_mean,s2_EVI_mean,s2_GNDVI_mean,s2_NDWI_GREEN_mean,s2_NDRE_mean,s2_NDMI_mean,s2_MSI_mean,s2_NBR_mean,s2_BSI_mean,s2_CAI_B01_B08_mean
min,-1.000000,-0.338368,-27.458071,-0.714893,-0.999920,-0.701307,-0.819733,0.170735,-0.798272,-0.659883,-0.500412
50%,0.216268,0.138864,0.129759,0.397614,-0.397614,0.137615,-0.126224,1.288916,-0.044073,0.196195,0.629729
max,1.000000,0.810555,38.113894,0.999920,0.714893,0.838539,0.708329,10.094664,0.863707,0.700404,0.999999


## Fetch Minimal Weather Features By Survey Zone

This section fetches historical daily weather from Open-Meteo using one representative centroid per Morocco survey zone. That keeps the download small while adding the same weather/GDD/rain/temp feature names used in the YieldSAT crop-mapping feature store.

In [15]:
import time
from datetime import timedelta

WEATHER_LOCATION_MODE = "survey_zone_median_centroid"
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

satellite_dates = pd.to_datetime(final_dataset["satellite_date"], errors="coerce")
weather_start_date = (satellite_dates.min() - pd.Timedelta(days=29)).date()
weather_cumulative_start_date = satellite_dates.min().date()
weather_end_date = satellite_dates.max().date()

weather_locations = (
    selected_parcel_metadata.dropna(subset=["centroid_lat", "centroid_lon"])
    .groupby("survey_zone", as_index=False)
    .agg(
        weather_location_lat=("centroid_lat", "median"),
        weather_location_lon=("centroid_lon", "median"),
        parcel_count=("parcel_id", "count"),
    )
)
weather_locations["weather_location_id"] = weather_locations["survey_zone"].astype(str)

print("Weather location mode:", WEATHER_LOCATION_MODE)
print("Weather date range:", weather_start_date, "to", weather_end_date)
print("Cumulative weather starts at:", weather_cumulative_start_date)
display(weather_locations)


def fetch_open_meteo_daily(latitude, longitude, start_date, end_date, timeout=90):
    params = {
        "latitude": float(latitude),
        "longitude": float(longitude),
        "start_date": str(start_date),
        "end_date": str(end_date),
        "daily": ",".join(
            [
                "temperature_2m_mean",
                "temperature_2m_max",
                "temperature_2m_min",
                "precipitation_sum",
            ]
        ),
        "timezone": "UTC",
    }
    response = requests.get(OPEN_METEO_URL, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    if "daily" not in payload:
        raise RuntimeError(f"Open-Meteo response did not include daily weather: {payload}")

    daily = pd.DataFrame(payload["daily"])
    daily = daily.rename(
        columns={
            "time": "date",
            "temperature_2m_mean": "weather_daily_Temp_mean_C",
            "temperature_2m_max": "weather_daily_Temp_max_C",
            "temperature_2m_min": "weather_daily_Temp_min_C",
            "precipitation_sum": "weather_daily_Total_prec_mm",
        }
    )
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    return daily


def add_weather_derivatives(daily_weather, cumulative_start_date):
    daily_weather = daily_weather.sort_values(["weather_location_id", "date"]).copy()
    daily_weather["weather_daily_gdd_base10"] = (daily_weather["weather_daily_Temp_mean_C"] - 10.0).clip(lower=0)
    daily_weather["weather_daily_rainy_day"] = daily_weather["weather_daily_Total_prec_mm"] >= 1.0
    daily_weather["weather_daily_heavy_rain_day"] = daily_weather["weather_daily_Total_prec_mm"] >= 20.0
    daily_weather["weather_daily_heat_stress_day"] = daily_weather["weather_daily_Temp_max_C"] >= 35.0
    daily_weather["weather_daily_cold_stress_day"] = daily_weather["weather_daily_Temp_min_C"] <= 5.0

    for window_days in (7, 14, 30):
        grouped = daily_weather.groupby("weather_location_id", group_keys=False)
        daily_weather[f"weather_{window_days}d_Total_prec_mm"] = grouped["weather_daily_Total_prec_mm"].rolling(window_days, min_periods=1).sum().reset_index(level=0, drop=True)
        daily_weather[f"weather_{window_days}d_Temp_mean_C"] = grouped["weather_daily_Temp_mean_C"].rolling(window_days, min_periods=1).mean().reset_index(level=0, drop=True)
        daily_weather[f"weather_{window_days}d_gdd_base10"] = grouped["weather_daily_gdd_base10"].rolling(window_days, min_periods=1).sum().reset_index(level=0, drop=True)
        daily_weather[f"weather_{window_days}d_rainy_days"] = grouped["weather_daily_rainy_day"].rolling(window_days, min_periods=1).sum().reset_index(level=0, drop=True)

    cumulative_start = pd.Timestamp(cumulative_start_date)
    cumulative_parts = []
    for location_id, part in daily_weather.groupby("weather_location_id", sort=False):
        part = part.copy()
        in_cumulative_window = part["date"] >= cumulative_start
        part["weather_cum_Total_prec_mm"] = np.nan
        part["weather_cum_gdd_base10"] = np.nan
        part["weather_cum_rainy_days"] = np.nan
        part["weather_cum_heavy_rain_days"] = np.nan
        part["weather_cum_heat_stress_days"] = np.nan
        part["weather_cum_cold_stress_days"] = np.nan

        if in_cumulative_window.any():
            part.loc[in_cumulative_window, "weather_cum_Total_prec_mm"] = part.loc[in_cumulative_window, "weather_daily_Total_prec_mm"].cumsum()
            part.loc[in_cumulative_window, "weather_cum_gdd_base10"] = part.loc[in_cumulative_window, "weather_daily_gdd_base10"].cumsum()
            part.loc[in_cumulative_window, "weather_cum_rainy_days"] = part.loc[in_cumulative_window, "weather_daily_rainy_day"].cumsum()
            part.loc[in_cumulative_window, "weather_cum_heavy_rain_days"] = part.loc[in_cumulative_window, "weather_daily_heavy_rain_day"].cumsum()
            part.loc[in_cumulative_window, "weather_cum_heat_stress_days"] = part.loc[in_cumulative_window, "weather_daily_heat_stress_day"].cumsum()
            part.loc[in_cumulative_window, "weather_cum_cold_stress_days"] = part.loc[in_cumulative_window, "weather_daily_cold_stress_day"].cumsum()
        cumulative_parts.append(part)

    return pd.concat(cumulative_parts, ignore_index=True)


weather_frames = []
for row in weather_locations.itertuples(index=False):
    print("Fetching weather:", row.weather_location_id, row.weather_location_lat, row.weather_location_lon)
    daily = fetch_open_meteo_daily(
        row.weather_location_lat,
        row.weather_location_lon,
        weather_start_date,
        weather_end_date,
    )
    daily["weather_location_id"] = row.weather_location_id
    daily["weather_location_lat"] = row.weather_location_lat
    daily["weather_location_lon"] = row.weather_location_lon
    daily["weather_location_mode"] = WEATHER_LOCATION_MODE
    weather_frames.append(daily)
    time.sleep(0.2)

weather_daily = pd.concat(weather_frames, ignore_index=True)
weather_daily = add_weather_derivatives(weather_daily, weather_cumulative_start_date)

print("Weather daily rows:", weather_daily.shape)
print("Weather columns:", weather_daily.columns.tolist())
display(weather_daily.head())


Weather location mode: survey_zone_median_centroid
Weather date range: 2023-12-03 to 2024-12-26
Cumulative weather starts at: 2024-01-01


,survey_zone,weather_location_lat,weather_location_lon,parcel_count,weather_location_id
0,Doukkala,32.699606,-8.587485,3948,Doukkala
1,Gharb,34.482189,-6.245757,1292,Gharb
2,Souss,30.566954,-8.661445,167,Souss
3,Statte,33.281066,-7.476379,248,Statte
4,Tadla,32.434250,-6.646416,1161,Tadla
5,el haouz,31.728001,-7.663954,918,el haouz


Fetching weather: Doukkala 32.69960559805896 -8.587485372794657
Fetching weather: Gharb 34.48218877787459 -6.245757094808596
Fetching weather: Souss 30.566953918321527 -8.661444926714156
Fetching weather: Statte 33.28106585752029 -7.476379125704652
Fetching weather: Tadla 32.43424981203959 -6.6464156255831695
Fetching weather: el haouz 31.72800105158118 -7.6639537998266585
Weather daily rows: (2340, 32)
Weather columns: ['date', 'weather_daily_Temp_mean_C', 'weather_daily_Temp_max_C', 'weather_daily_Temp_min_C', 'weather_daily_Total_prec_mm', 'weather_location_id', 'weather_location_lat', 'weather_location_lon', 'weather_location_mode', 'weather_daily_gdd_base10', 'weather_daily_rainy_day', 'weather_daily_heavy_rain_day', 'weather_daily_heat_stress_day', 'weather_daily_cold_stress_day', 'weather_7d_Total_prec_mm', 'weather_7d_Temp_mean_C', 'weather_7d_gdd_base10', 'weather_7d_rainy_days', 'weather_14d_Total_prec_mm', 'weather_14d_Temp_mean_C', 'weather_14d_gdd_base10', 'weather_14d_rai

,date,weather_daily_Temp_mean_C,weather_daily_Temp_max_C,weather_daily_Temp_min_C,weather_daily_Total_prec_mm,weather_location_id,weather_location_lat,weather_location_lon,weather_location_mode,weather_daily_gdd_base10,...,weather_30d_Total_prec_mm,weather_30d_Temp_mean_C,weather_30d_gdd_base10,weather_30d_rainy_days,weather_cum_Total_prec_mm,weather_cum_gdd_base10,weather_cum_rainy_days,weather_cum_heavy_rain_days,weather_cum_heat_stress_days,weather_cum_cold_stress_days
0,2023-12-03,14.2,21.1,9.6,0.0,Doukkala,32.699606,-8.587485,survey_zone_median_centroid,4.2,...,0.0,14.20,4.2,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-12-04,14.1,23.0,7.3,0.0,Doukkala,32.699606,-8.587485,survey_zone_median_centroid,4.1,...,0.0,14.15,8.3,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-12-05,17.0,23.1,11.6,7.8,Doukkala,32.699606,-8.587485,survey_zone_median_centroid,7.0,...,7.8,15.10,15.3,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-12-06,16.9,20.5,13.8,0.7,Doukkala,32.699606,-8.587485,survey_zone_median_centroid,6.9,...,8.5,15.55,22.2,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-12-07,17.1,20.1,14.1,0.3,Doukkala,32.699606,-8.587485,survey_zone_median_centroid,7.1,...,8.8,15.86,29.3,1.0,NaN,NaN,NaN,NaN,NaN,NaN


## Join Weather Features

This step joins weather features into the aligned Morocco observations in memory. The final enriched dataset is saved once after parcel metadata is attached.


In [16]:
aligned_observations_with_weather = aligned_observations.copy()
aligned_observations_with_weather["weather_location_id"] = aligned_observations_with_weather["survey_zone"].astype(str)
aligned_observations_with_weather["date"] = pd.to_datetime(aligned_observations_with_weather["date"], errors="coerce")
weather_for_join = weather_daily.copy()
weather_for_join["date"] = pd.to_datetime(weather_for_join["date"], errors="coerce")

weather_feature_columns = [
    col for col in weather_for_join.columns
    if col.startswith("weather_") and col not in {"weather_location_id", "weather_location_mode", "weather_location_lat", "weather_location_lon"}
]
weather_join_columns = ["weather_location_id", "date"] + weather_feature_columns

aligned_observations_with_weather = aligned_observations_with_weather.merge(
    weather_for_join[weather_join_columns],
    on=["weather_location_id", "date"],
    how="left",
)

for col in aligned_observations_with_weather.columns:
    if pd.api.types.is_bool_dtype(aligned_observations_with_weather[col]):
        aligned_observations_with_weather[col] = aligned_observations_with_weather[col].astype("int8")

print("Aligned observations with weather shape:", aligned_observations_with_weather.shape)
print("Weather feature columns joined:", len(weather_feature_columns))
print("Missing weather rows:", aligned_observations_with_weather["weather_daily_Temp_mean_C"].isna().sum())
print("Observation feature groups so far:")
print({
    "sentinel2_band_means": len([col for col in aligned_observations_with_weather.columns if col.startswith("s2_B")]),
    "vegetation_index_means": len([col for col in aligned_observations_with_weather.columns if any(col.startswith(f"s2_{name}") for name in VEGETATION_INDEX_NAMES)]),
    "weather": len(weather_feature_columns),
})


Aligned observations with weather shape: (390766, 57)
Weather feature columns joined: 27
Missing weather rows: 0
Observation feature groups so far:
{'sentinel2_band_means': 13, 'vegetation_index_means': 11, 'weather': 27}


## Optional Future SCL Quality Features

SCL/cloud-quality features are not generated in this notebook because the full pass is expensive and the YieldSAT exploration showed SCL was a low-importance feature group compared with Sentinel-2 bands, vegetation indices, and weather. If SCL is needed later, use Google Earth Engine with `ee.Filter.listContains("system:band_names", "SCL")`; do not use `ee.Filter.notNull(["SCL"])`, because `SCL` is a band name, not an image metadata key. Earth Engine `frequencyHistogram()` can return fractional edge-pixel counts, so compute fractions from those weighted counts rather than assuming integer pixels.


In [17]:
FEATURES_OUTPUT_PATH = DATA_DIR / "features.parquet"

final_features = aligned_observations_with_weather.copy()
final_features["date"] = pd.to_datetime(final_features["date"], errors="coerce")

metadata_columns = [
    "parcel_id",
    "label",
    "model_label",
    "survey_date",
    "irrigation_type",
    "centroid_lon",
    "centroid_lat",
    "bbox_min_lon",
    "bbox_min_lat",
    "bbox_max_lon",
    "bbox_max_lat",
    "area_ha",
    "country",
    "year",
]

final_features = final_features.merge(
    selected_parcel_metadata[metadata_columns],
    on="parcel_id",
    how="left",
)

final_features["satellite_date"] = final_features["date"].dt.date.astype(str)
final_features = final_features.drop(columns=["date"])

front_columns = [
    "field_id",
    "parcel_id",
    "satellite_date",
    "label",
    "model_label",
    "survey_date",
    "irrigation_type",
    "survey_zone",
    "country",
    "year",
    "dataset_source",
]
remaining_columns = [col for col in final_features.columns if col not in front_columns]
final_features = final_features[front_columns + remaining_columns]

for col in final_features.columns:
    if pd.api.types.is_bool_dtype(final_features[col]):
        final_features[col] = final_features[col].astype("int8")

final_features.to_parquet(FEATURES_OUTPUT_PATH, index=False)

print("Saved final enriched features:", FEATURES_OUTPUT_PATH)
print("Final features shape:", final_features.shape)
print("Output files kept:")
print("-", DATA_DIR / "raw_merged.parquet")
print("-", DATA_DIR / "morocco_selected_crop_background_timeseries.parquet")
print("-", FEATURES_OUTPUT_PATH)
print("Feature groups in final file:")
print({
    "sentinel2_band_means": len([col for col in final_features.columns if col.startswith("s2_B")]),
    "vegetation_index_means": len([col for col in final_features.columns if any(col.startswith(f"s2_{name}") for name in VEGETATION_INDEX_NAMES)]),
    "weather": len([col for col in final_features.columns if col.startswith("weather_")]),
    "geometry_metadata": len([col for col in ["centroid_lon", "centroid_lat", "bbox_min_lon", "bbox_min_lat", "bbox_max_lon", "bbox_max_lat", "area_ha"] if col in final_features.columns]),
})


Saved final enriched features: data/morocco_crop_irrigation/features.parquet
Final features shape: (390766, 70)
Output files kept:
- data/morocco_crop_irrigation/raw_merged.parquet
- data/morocco_crop_irrigation/morocco_selected_crop_background_timeseries.parquet
- data/morocco_crop_irrigation/features.parquet
Feature groups in final file:
{'sentinel2_band_means': 13, 'vegetation_index_means': 11, 'weather': 28, 'geometry_metadata': 7}
